In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import cv2
import os
import re
from pathlib import Path
from scipy.optimize import least_squares

IMG_WIDTH, IMG_HEIGHT = 1920, 1080
VIDEO_FPS = 60

TELEMETRY_CSV = "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-00-04-csv/DJIFlightRecord_2026-04-02_17-00-04.csv"
VIDEO_OFFSET_S = -2.62
FRAMES_DIR = "/kaggle/input/datasets/natair/chickens-drone-25k/chickens_drone_25k"
TRACKS_XML = "/kaggle/input/datasets/natair/chicken-trainings-data/tracks_dense.xml"

SHELTER_HEIGHT_M = 1.6

In [ ]:
# Telemetry + Geodesy

telemetry = pd.read_csv(TELEMETRY_CSV)[["time_s", "lat", "lng", "altitude_m", "yaw_deg"]].dropna().reset_index(drop=True)
REF_LAT, REF_LNG = telemetry["lat"].mean(), telemetry["lng"].mean()

def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    return (lng - ref_lng) * m_per_deg_lng, (lat - ref_lat) * m_per_deg_lat

def local_m_to_latlng(east, north, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    return ref_lat + north / m_per_deg_lat, ref_lng + east / m_per_deg_lng

def get_drone_state_interp(t):
    ts = telemetry["time_s"].values
    lat = np.interp(t, ts, telemetry["lat"].values)
    lng = np.interp(t, ts, telemetry["lng"].values)
    alt = np.interp(t, ts, telemetry["altitude_m"].values)

    yaw_unwrapped = np.unwrap(np.radians(telemetry["yaw_deg"].values))
    yaw = np.degrees(np.interp(t, ts, yaw_unwrapped)) % 360
    return lat, lng, alt, yaw

def get_drone_state(frame):
    t = frame / VIDEO_FPS + VIDEO_OFFSET_S
    return get_drone_state_interp(t)

print(f" Telemetry loaded: {len(telemetry)} lines, reference point lat={REF_LAT:.6f}, lng={REF_LNG:.6f}")


In [ ]:
# Full camera beam model (tilt taken into account)

def project_ray_to_height(drone_east, drone_north, height, yaw_deg, pitch_down_deg, focal_px, px, py, target_height=0.0):
    # Pixel -> point on the target_height plane (in local meters)
    yaw, pitch = np.radians(yaw_deg), np.radians(pitch_down_deg)
    forward = np.array([np.sin(yaw)*np.cos(pitch), np.cos(yaw)*np.cos(pitch), -np.sin(pitch)])
    right = np.array([np.cos(yaw), -np.sin(yaw), 0])
    down = np.cross(forward, right)
    dx, dy = px - IMG_WIDTH/2, py - IMG_HEIGHT/2
    d = right*(dx/focal_px) + down*(dy/focal_px) + forward
    d = d/np.linalg.norm(d)
    if d[2] >= -1e-6:
        return None
    t = (target_height - height) / d[2]
    return drone_east + t*d[0], drone_north + t*d[1]

def world_to_pixel(drone_east, drone_north, height, yaw_deg, pitch_down_deg, focal_px, X, Y, Z):
    # Point in local meters -> pixel (inverse projection, for reprojection error)
    yaw, pitch = np.radians(yaw_deg), np.radians(pitch_down_deg)
    forward = np.array([np.sin(yaw)*np.cos(pitch), np.cos(yaw)*np.cos(pitch), -np.sin(pitch)])
    right = np.array([np.cos(yaw), -np.sin(yaw), 0])
    down = np.cross(forward, right)
    rel = np.array([X-drone_east, Y-drone_north, Z-height])
    lf, lr, ld = rel@forward, rel@right, rel@down
    if lf <= 1e-6:
        return None
    return IMG_WIDTH/2 + focal_px*lr/lf, IMG_HEIGHT/2 + focal_px*ld/lf

In [ ]:
def frame_number(path):
    match = re.search(r'(\d+)', path.stem)
    return int(match.group(1)) if match else 0

EXTS = {'.jpg', '.jpeg', '.png'}
frame_files = sorted(
    [p for p in Path(FRAMES_DIR).rglob("*") if p.suffix.lower() in EXTS],
    key=frame_number
)
print(f" Frames found: {len(frame_files)}")

def contact_sheet(frame_indices, cols=5):
    valid_indices = [idx for idx in frame_indices if idx < len(frame_files)]
    if not valid_indices:
        print("None of the specified indexes are in the frame_files list")
        return

    rows = (len(valid_indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
    axes = axes.flat if rows > 1 else ([axes] if cols == 1 else axes)

    for ax, idx in zip(axes, valid_indices):
        img_bgr = cv2.imread(str(frame_files[idx]))

        if img_bgr is None:
            ax.text(0.5, 0.5, f"Could not read frame {idx}", color='red', ha='center', va='center')
            ax.axis("off")
            continue

        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Frame {idx}", fontsize=9)
        ax.axis("off")

    for ax in list(axes)[len(valid_indices):]:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("/kaggle/working/contact_sheet.jpg", dpi=100, bbox_inches="tight")
    plt.show()
    plt.close('all')

def show_grid(frame_idx, step=100):
    fp = frame_files[frame_idx]
    img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img)
    for x in range(0, W, step):
        ax.axvline(x, color='yellow', alpha=0.25, linewidth=0.5)
        ax.text(x, 15, str(x), color='red', fontsize=7)
    for y in range(0, H, step):
        ax.axhline(y, color='yellow', alpha=0.25, linewidth=0.5)
        ax.text(5, y, str(y), color='red', fontsize=7)
    ax.set_title(f"Frame {frame_idx}")
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/grid_frame_{frame_idx}.jpg", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
show_grid(5000)

In [ ]:
import base64
from IPython.display import HTML

def click_to_coords(frame_idx):
    img_path = str(frame_files[frame_idx])
    with open(img_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    html = f"""
    <div style="font-family: monospace;">
      <div style="margin-bottom:8px; font-size:14px;">Frame {frame_idx}</div>
      <img id="clkimg_{frame_idx}" src="data:image/jpeg;base64,{img_b64}"
           style="max-width:100%; cursor:crosshair; border:1px solid #888;" />
      <div id="log_{frame_idx}" style="margin-top:10px; font-size:14px; line-height:1.6;"></div>
      <button id="reset_{frame_idx}" style="margin-top:8px; padding:4px 12px; cursor:pointer;">
        Clear
      </button>
    </div>
    <script>
    (function() {{
        var img = document.getElementById('clkimg_{frame_idx}');
        var log = document.getElementById('log_{frame_idx}');
        var resetBtn = document.getElementById('reset_{frame_idx}');
        var clicks = [];

        img.onclick = function(e) {{
            var rect = img.getBoundingClientRect();
            var scaleX = img.naturalWidth / rect.width;
            var scaleY = img.naturalHeight / rect.height;
            var x = Math.round((e.clientX - rect.left) * scaleX);
            var y = Math.round((e.clientY - rect.top) * scaleY);
            clicks.push([x, y]);
            renderLog();
        }};

        resetBtn.onclick = function() {{
            clicks = [];
            renderLog();
        }};

        function renderLog() {{
            var lines = clicks.map(function(c, i) {{
                return "Dot " + (i+1) + ":  px=" + c[0] + ",  py=" + c[1];
            }});
            log.innerHTML = lines.join("<br>");
        }}
    }})();
    </script>
    """
    return HTML(html)

click_to_coords(25300)

In [ ]:
# Final projection: pixel → GPS (full model, dx and dy)

def pixel_to_latlng(px, py, frame):
    lat, lng, h, yaw = get_drone_state(frame)
    de, dn = latlng_to_local_m(lat, lng, REF_LAT, REF_LNG)
    proj = project_ray_to_height(de, dn, h, yaw, PITCH_DOWN_DEG, FOCAL_PX, px, py, target_height=0)
    if proj is None:
        return None
    return local_m_to_latlng(proj[0], proj[1], REF_LAT, REF_LNG)


In [ ]:
SHELTER_SIGHTINGS = [
    # {"frame": 1234, "base_px": 812, "base_py": 640, "top_px": 799, "top_py": 590},
    {"frame": 1, "base_px": 1265, "base_py": 567, "top_px": 1262, "top_py": 470},
    {"frame": 1, "base_px": 719, "base_py": 534, "top_px": 725, "top_py": 460},
    {"frame": 100, "base_px": 1129, "base_py": 524, "top_px": 1128, "top_py": 449},
    {"frame": 100, "base_px": 1715, "base_py": 573, "top_px": 1715, "top_py": 469},
    {"frame": 200, "base_px": 1827, "base_py": 680, "top_px": 1827, "top_py": 556},
    {"frame": 200, "base_px": 1210, "base_py": 617, "top_px": 1215, "top_py": 527},
    {"frame": 300, "base_px": 1098, "base_py": 677, "top_px": 1098, "top_py": 586},
    {"frame": 300, "base_px": 1684, "base_py": 758, "top_px": 1684, "top_py": 640},
    {"frame": 400, "base_px": 1034, "base_py": 606, "top_px": 1034, "top_py": 504},
    {"frame": 400, "base_px": 490, "base_py": 534, "top_px": 490, "top_py": 447},
    {"frame": 500, "base_px": 1088, "base_py": 696, "top_px": 1088, "top_py": 596},
    {"frame": 500, "base_px": 554, "base_py": 606, "top_px": 554, "top_py": 524},
    {"frame": 600, "base_px": 1108, "base_py": 698, "top_px": 1108, "top_py": 599},
    {"frame": 600, "base_px": 584, "base_py": 582, "top_px": 584, "top_py": 501},
    {"frame": 600, "base_px": 303, "base_py": 428, "top_px": 303, "top_py": 373},
    {"frame": 600, "base_px": 225, "base_py": 394, "top_px": 225, "top_py": 347},
    {"frame": 700, "base_px": 1111, "base_py": 733, "top_px": 1111, "top_py": 631},
    {"frame": 700, "base_px": 597, "base_py": 593, "top_px": 597, "top_py": 513},
    {"frame": 700, "base_px": 315, "base_py": 406, "top_px": 315, "top_py": 351},
    {"frame": 700, "base_px": 237, "base_py": 362, "top_px": 237, "top_py": 316},
    {"frame": 800, "base_px": 1094, "base_py": 667, "top_px": 1094, "top_py": 579},
    {"frame": 800, "base_px": 591, "base_py": 518, "top_px": 591, "top_py": 444},
    {"frame": 800, "base_px": 301, "base_py": 292, "top_px": 301, "top_py": 240},
    {"frame": 800, "base_px": 219, "base_py": 249, "top_px": 219, "top_py": 200},
    {"frame": 900, "base_px": 1098, "base_py": 774, "top_px": 1098, "top_py": 693},
    {"frame": 900, "base_px": 607, "base_py": 617, "top_px": 607, "top_py": 545},
    {"frame": 900, "base_px": 324, "base_py": 365, "top_px": 324, "top_py": 316},
    {"frame": 900, "base_px": 243, "base_py": 318, "top_px": 243, "top_py": 272},
    {"frame": 1000, "base_px": 1060, "base_py": 715, "top_px": 1060, "top_py": 635},
    {"frame": 1000, "base_px": 580, "base_py": 550, "top_px": 580, "top_py": 479},
    {"frame": 1000, "base_px": 288, "base_py": 272, "top_px": 288, "top_py": 218},
    {"frame": 1000, "base_px": 205, "base_py": 217, "top_px": 205, "top_py": 166},
    {"frame": 1100, "base_px": 1071, "base_py": 756, "top_px": 1071, "top_py": 681},
    {"frame": 1100, "base_px": 609, "base_py": 580, "top_px": 609, "top_py": 512},
    {"frame": 1100, "base_px": 320, "base_py": 276, "top_px": 320, "top_py": 226},
    {"frame": 1100, "base_px": 236, "base_py": 220, "top_px": 236, "top_py": 171},
    {"frame": 1500, "base_px": 1374, "base_py": 1039, "top_px": 1374, "top_py": 964},
    {"frame": 1500, "base_px": 971, "base_py": 738, "top_px": 971, "top_py": 672},
    {"frame": 1500, "base_px": 815, "base_py": 379, "top_px": 815, "top_py": 333},
    {"frame": 1500, "base_px": 762, "base_py": 310, "top_px": 762, "top_py": 267},
    {"frame": 1800, "base_px": 1095, "base_py": 904, "top_px": 1095, "top_py": 834},
    {"frame": 1800, "base_px": 883, "base_py": 623, "top_px": 883, "top_py": 564},
    {"frame": 1800, "base_px": 915, "base_py": 324, "top_px": 915, "top_py": 284},
    {"frame": 1800, "base_px": 900, "base_py": 263, "top_px": 900, "top_py": 227},
    {"frame": 2100, "base_px": 1114, "base_py": 677, "top_px": 1114, "top_py": 605},
    {"frame": 2100, "base_px": 1579, "base_py": 383, "top_px": 1579, "top_py": 331},
    {"frame": 2100, "base_px": 1639, "base_py": 316, "top_px": 1639, "top_py": 269},
    {"frame": 2500, "base_px": 546, "base_py": 768, "top_px": 546, "top_py": 675},
    {"frame": 2500, "base_px": 1072, "base_py": 550, "top_px": 1072, "top_py": 469},
    {"frame": 2800, "base_px": 165, "base_py": 1004, "top_px": 165, "top_py": 894},
    {"frame": 2800, "base_px": 1167, "base_py": 483, "top_px": 1167, "top_py": 383},
    {"frame": 3100, "base_px": 1213, "base_py": 1022, "top_px": 1245, "top_py": 942},
    {"frame": 3500, "base_px": 1210, "base_py": 921, "top_px": 1245, "top_py": 828},
    {"frame": 3700, "base_px": 1232, "base_py": 898, "top_px": 1232, "top_py": 814},
    {"frame": 4150, "base_px": 1531, "base_py": 255, "top_px": 1567, "top_py": 152},
    {"frame": 4500, "base_px": 968, "base_py": 233, "top_px": 976, "top_py": 125},
    {"frame": 4800, "base_px": 776, "base_py": 434, "top_px": 774, "top_py": 333},
    {"frame": 5100, "base_px": 1354, "base_py": 458, "top_px": 1331, "top_py": 354},
    {"frame": 5500, "base_px": 1513, "base_py": 466, "top_px": 1534, "top_py": 357},
    {"frame": 6200, "base_px": 1037, "base_py": 623, "top_px": 1054, "top_py": 510},
    {"frame": 6700, "base_px": 381, "base_py": 866, "top_px": 337, "top_py": 761},
    {"frame": 6850, "base_px": 404, "base_py": 912, "top_px": 381, "top_py": 813},
    {"frame": 9800, "base_px": 1562, "base_py": 1060, "top_px": 1635, "top_py": 1054},
    {"frame": 10000, "base_px": 1005, "base_py": 828, "top_px": 1019, "top_py": 718},
    {"frame": 10500, "base_px": 681, "base_py": 767, "top_px": 681, "top_py": 652},
    {"frame": 11000, "base_px": 678, "base_py": 785, "top_px": 678, "top_py": 663},
    {"frame": 11500, "base_px": 999, "base_py": 799, "top_px": 999, "top_py": 687},
    {"frame": 11700, "base_px": 451, "base_py": 522, "top_px": 451, "top_py": 400},
    {"frame": 11800, "base_px": 370, "base_py": 706, "top_px": 370, "top_py": 608},
    {"frame": 12500, "base_px": 818, "base_py": 447, "top_px": 818, "top_py": 340},
    {"frame": 12800, "base_px": 942, "base_py": 360, "top_px": 942, "top_py": 233},
    {"frame": 13100, "base_px": 942, "base_py": 661, "top_px": 942, "top_py": 493},
    {"frame": 21700, "base_px": 1152, "base_py": 1005, "top_px": 1169, "top_py": 939},
    {"frame": 22000, "base_px": 1141, "base_py": 322, "top_px": 1155, "top_py": 233},
    {"frame": 22000, "base_px": 386, "base_py": 470, "top_px": 373, "top_py": 385},
    {"frame": 22500, "base_px": 1135, "base_py": 304, "top_px": 1149, "top_py": 223},
    {"frame": 22500, "base_px": 378, "base_py": 454, "top_px": 366, "top_py": 373},
    {"frame": 23000, "base_px": 1135, "base_py": 305, "top_px": 1146, "top_py": 220},
    {"frame": 23000, "base_px": 372, "base_py": 460, "top_px": 364, "top_py": 373},
    {"frame": 23500, "base_px": 1008, "base_py": 298, "top_px": 1008, "top_py": 217},
    {"frame": 23500, "base_px": 217, "base_py": 454, "top_px": 194, "top_py": 368},
    {"frame": 24000, "base_px": 555, "base_py": 597, "top_px": 549, "top_py": 513},
    {"frame": 24000, "base_px": 1296, "base_py": 660, "top_px": 1314, "top_py": 585},
    {"frame": 24500, "base_px": 545, "base_py": 483, "top_px": 545, "top_py": 398},
    {"frame": 24500, "base_px": 1305, "base_py": 548, "top_px": 1305, "top_py": 467},
    {"frame": 25000, "base_px": 548, "base_py": 498, "top_px": 543, "top_py": 415},
    {"frame": 25000, "base_px": 1297, "base_py": 567, "top_px": 1316, "top_py": 481},
    {"frame": 25300, "base_px": 1053, "base_py": 640, "top_px": 1062, "top_py": 556},
    {"frame": 25300, "base_px": 194, "base_py": 725, "top_px": 168, "top_py": 631},
]

if len(SHELTER_SIGHTINGS) < 5:
    print(f"Not enough {len(SHELTER_SIGHTINGS)}")
else:
    for s in SHELTER_SIGHTINGS:
        lat, lng, h, yaw = get_drone_state(s["frame"])
        s["height_m"], s["yaw_deg"] = h, yaw

    def calib_residuals(params, obs):
        pitch, focal = params
        errs = []
        for o in obs:
            base_xy = project_ray_to_height(0, 0, o["height_m"], o["yaw_deg"], pitch, focal, o["base_px"], o["base_py"], 0)
            if base_xy is None:
                errs.extend([50, 50]); continue
            pred_top = world_to_pixel(0, 0, o["height_m"], o["yaw_deg"], pitch, focal, base_xy[0], base_xy[1], SHELTER_HEIGHT_M)
            if pred_top is None:
                errs.extend([50, 50]); continue
            errs.extend([pred_top[0]-o["top_px"], pred_top[1]-o["top_py"]])
        return errs

    result = least_squares(calib_residuals, x0=[50.0, 1200.0], args=(SHELTER_SIGHTINGS,), bounds=([5,300],[89,4000]))
    PITCH_DOWN_DEG, FOCAL_PX = result.x
    print(f"pitch_down = {PITCH_DOWN_DEG:.1f}°  (90°=nadir)")
    print(f"focal_px   = {FOCAL_PX:.0f}px")

In [ ]:
# Applying to tracks_dense.xml

if not os.path.exists(TRACKS_XML):
    print(f"{TRACKS_XML} not found")
elif "PITCH_DOWN_DEG" not in dir():
    print("Need calibrations pitch/focal")
else:
    tree = ET.parse(TRACKS_XML)
    root = tree.getroot()
    rows = []
    for track in root.findall(".//track"):
        track_id = track.get("id")
        for box in track.findall("box"):
            if box.get("outside") == "1":
                continue
            frame = int(box.get("frame"))
            cx = (float(box.get("xtl")) + float(box.get("xbr"))) / 2
            cy = (float(box.get("ytl")) + float(box.get("ybr"))) / 2
            latlng = pixel_to_latlng(cx, cy, frame)
            if latlng is None:
                continue
            lat, lon = latlng
            rows.append({"track_id": track_id, "frame": frame, "time_s": frame / VIDEO_FPS, "lat": lat, "lon": lon})

    df_geo = pd.DataFrame(rows)
    df_geo.to_csv("/kaggle/working/chicken_positions_v4.csv", index=False)
    print(f"{len(df_geo)} geolocated detections, {df_geo[‘track_id’].nunique() if len(df_geo) else 0} tracks")


In [ ]:
# Validation

CSV_PATH = "/kaggle/working/chicken_positions_v4.csv"

if os.path.exists(CSV_PATH):
    df_geo = pd.read_csv(CSV_PATH)
    east_t, north_t = latlng_to_local_m(telemetry["lat"].values, telemetry["lng"].values, REF_LAT, REF_LNG)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(east_t, north_t, "-", color="lightgray", linewidth=1, label="drone track")
    for tid, grp in df_geo.groupby("track_id"):
        e, n = latlng_to_local_m(grp["lat"].values, grp["lon"].values, REF_LAT, REF_LNG)
        ax.plot(e, n, "o-", markersize=3, linewidth=1, alpha=0.7)
    ax.set_xlabel("East, m"); ax.set_ylabel("North, m"); ax.set_aspect("equal")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("/kaggle/working/validation_v4.png", dpi=110); plt.show()

    speeds = []
    for tid, grp in df_geo.sort_values("time_s").groupby("track_id"):
        e, n = latlng_to_local_m(grp["lat"].values, grp["lon"].values, REF_LAT, REF_LNG)
        dt = np.diff(grp["time_s"].values)
        dist = np.hypot(np.diff(e), np.diff(n))
        valid = dt > 0
        speeds.extend((dist[valid]/dt[valid]).tolist())
    if speeds:
        print(f"Speed: median={np.median(speeds):.2f} m/s, 95th percentile={np.percentile(speeds,95):.2f} m/s")
else:
    print("chicken_positions_v4.csv not found")

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import os

if not os.path.exists(TRACKS_XML):
    print(f"{TRACKS_XML} not found")
else:
    tree = ET.parse(TRACKS_XML)
    root = tree.getroot()
    rows = []
    for track in root.findall(".//track"):
        track_id = track.get("id")
        for box in track.findall("box"):
            if box.get("outside") == "1":
                continue
            frame = int(box.get("frame"))
            cx = (float(box.get("xtl")) + float(box.get("xbr"))) / 2
            cy = (float(box.get("ytl")) + float(box.get("ybr"))) / 2

            latlng = pixel_to_latlng(cx, cy, frame)
            if latlng is None:
                continue
            lat, lon = latlng

            rows.append({
                "track_id": track_id,
                "frame": frame,
                "time_s": frame / VIDEO_FPS,
                "base_px": cx,
                "base_py": cy,
                "lat": lat,
                "lon": lon
            })

    df_geo = pd.DataFrame(rows)
    print("Column Check:", df_geo.columns.tolist())
    print(f"A total of {len(df_geo)} records were successfully loaded.")

In [ ]:
# Detection of "unexpected drift" using optical flow
import cv2

def sparse_flow_shift_robust(frame_idx_a, frame_idx_b, max_corners=200):
    img_a = cv2.cvtColor(cv2.imread(str(frame_files[frame_idx_a])), cv2.COLOR_BGR2GRAY)
    img_b = cv2.cvtColor(cv2.imread(str(frame_files[frame_idx_b])), cv2.COLOR_BGR2GRAY)
    pts_a = cv2.goodFeaturesToTrack(img_a, maxCorners=max_corners, qualityLevel=0.01, minDistance=20)
    if pts_a is None:
        return None
    pts_b, status, err = cv2.calcOpticalFlowPyrLK(img_a, img_b, pts_a, None)
    good_a = pts_a[status.flatten() == 1]
    good_b = pts_b[status.flatten() == 1]
    if len(good_a) < 15:
        return None

    # Affine model using RANSAC: filters out points that cause parallax (points elevated above the ground) as outliers
    M, inlier_mask = cv2.estimateAffinePartial2D(good_a, good_b, method=cv2.RANSAC,
                                                   ransacReprojThreshold=2.0)
    if M is None:
        return None
    inliers_b = good_b[inlier_mask.flatten() == 1]
    inliers_a = good_a[inlier_mask.flatten() == 1]
    shift = inliers_b - inliers_a
    return shift.reshape(-1, 2), inlier_mask.sum(), len(good_a)

def expected_shift_from_telemetry(frame_a, frame_b, pitch_guess, focal_guess, sample_px=(IMG_WIDTH/2, IMG_HEIGHT/2)):
    lat_a, lng_a, h_a, yaw_a = get_drone_state(frame_a)
    lat_b, lng_b, h_b, yaw_b = get_drone_state(frame_b)
    de_a, dn_a = latlng_to_local_m(lat_a, lng_a, REF_LAT, REF_LNG)
    de_b, dn_b = latlng_to_local_m(lat_b, lng_b, REF_LAT, REF_LNG)
    world_pt = project_ray_to_height(de_a, dn_a, h_a, yaw_a, pitch_guess, focal_guess,
                                       sample_px[0], sample_px[1], 0)
    if world_pt is None:
        return None
    pred_px_b = world_to_pixel(de_b, dn_b, h_b, yaw_b, pitch_guess, focal_guess,
                                 world_pt[0], world_pt[1], 0)
    if pred_px_b is None:
        return None
    dx, dy = pred_px_b[0] - sample_px[0], pred_px_b[1] - sample_px[1]

    if abs(dx) > IMG_WIDTH or abs(dy) > IMG_HEIGHT:
        return None
    return dx, dy

In [ ]:
HORIZONTAL_FOV_DEG = 85.0
FOCAL_PX_FIXED = (IMG_WIDTH / 2) / np.tan(np.radians(HORIZONTAL_FOV_DEG / 2))
print(f"Fixed focal length (as specified): {FOCAL_PX_FIXED:.0f}px")

In [ ]:
STEP = 100
anomaly = []
for i in range(0, len(frame_files) - 1):
    fa = frame_number(frame_files[i])
    fb = frame_number(frame_files[i+1])
    if fb - fa > 500:
        continue
    if fa % STEP != 0:
        continue

    flow_result = sparse_flow_shift_robust(i, i+1)  # Version with RANSAC
    if flow_result is None:
        continue
    shifts, inlier_count, total_count = flow_result
    if inlier_count < 10:
        continue
    actual_dx, actual_dy = np.median(shifts[:, 0]), np.median(shifts[:, 1])

    exp = expected_shift_from_telemetry(fa, fb, PITCH_DOWN_DEG, FOCAL_PX)
    if exp is None:
        continue
    exp_dx, exp_dy = exp

    residual = np.hypot(actual_dx - exp_dx, actual_dy - exp_dy)
    anomaly.append({"frame": fa, "actual": (actual_dx, actual_dy),
                     "expected": (exp_dx, exp_dy), "residual": residual})

df_anom = pd.DataFrame(anomaly).sort_values("residual", ascending=False)
print(df_anom.head(30))

In [ ]:
HORIZONTAL_FOV_DEG = 84.0
FOCAL_PX_FIXED = (IMG_WIDTH / 2) / np.tan(np.radians(HORIZONTAL_FOV_DEG / 2))
print(f"Fixed focal length: {FOCAL_PX_FIXED:.0f}px")

In [ ]:
# Knot frame recommendation based on anomaly density
def suggest_knot_frames(df_anom, n_knots=15, min_gap=300):
    candidates = df_anom.sort_values("residual", ascending=False)["frame"].tolist()
    knots = []
    for f in candidates:
        if all(abs(f - k) > min_gap for k in knots):
            knots.append(f)
        if len(knots) >= n_knots:
            break
    return sorted(knots)

KNOT_FRAMES = sorted(set([
    1, 200, 400, 600,
    2300, 2400,
    3800, 4000,
    5800, 6000, 6200, 6400, 6500, 6600,
    6900, 7000, 7600, 7900,
    10000, 10100,
    11700,
    13200, 13500, 13700,
    21600, 21700,
    25100, 25300
]))
print(f"Total number of knots: {len(KNOT_FRAMES)}")
print(KNOT_FRAMES)

In [ ]:
# Calibration with square-dependent pitch
def pitch_at(frame, knot_frames, knot_pitches):
    return np.interp(frame, knot_frames, knot_pitches)

# Tighten the bounds a bit and penalize sudden jumps between adjacent knots on the pitch
def calib_residuals_piecewise(params, obs, knot_frames, smoothness_weight=0.5):
    knot_pitches = params[:-1]
    focal = params[-1]
    errs = []
    for o in obs:
        pitch = pitch_at(o["frame"], knot_frames, knot_pitches)
        base_xy = project_ray_to_height(0, 0, o["height_m"], o["yaw_deg"], pitch, focal,
                                         o["base_px"], o["base_py"], 0)
        if base_xy is None:
            errs.extend([50, 50]); continue
        pred_top = world_to_pixel(0, 0, o["height_m"], o["yaw_deg"], pitch, focal,
                                   base_xy[0], base_xy[1], SHELTER_HEIGHT_M)
        if pred_top is None:
            errs.extend([50, 50]); continue
        errs.extend([pred_top[0]-o["top_px"], pred_top[1]-o["top_py"]])
    # Penalize differences in adjacent knot pitches
    diffs = np.diff(knot_pitches)
    errs.extend((smoothness_weight * diffs).tolist())
    return errs

for s in SHELTER_SIGHTINGS:
    if "height_m" not in s:
        lat, lng, h, yaw = get_drone_state(s["frame"])
        s["height_m"], s["yaw_deg"] = h, yaw

lo = [5]*len(KNOT_FRAMES) + [FOCAL_PX_FIXED*0.9]
hi = [89]*len(KNOT_FRAMES) + [FOCAL_PX_FIXED*1.1]
x0 = [45.0]*len(KNOT_FRAMES) + [FOCAL_PX_FIXED]

result2 = least_squares(calib_residuals_piecewise, x0=x0,
                          args=(SHELTER_SIGHTINGS, KNOT_FRAMES), bounds=(lo, hi))
*KNOT_PITCHES, FOCAL_PX = result2.x

print("Square-dependent pitch values:")
for f, p in zip(KNOT_FRAMES, KNOT_PITCHES):
    print(f"  square {f:>6}: pitch = {p:.1f}°")
print(f"focal_px = {FOCAL_PX:.0f}px")

In [ ]:
def sparse_flow_shift_matrix(img_path_a, img_path_b, max_corners=300):
    """
    It calculates the ground motion between two frames and, using RANSAC,
    filters out parallax (tall objects) to return the background affine matrix.
    """
    img_a = cv2.imread(str(img_path_a), cv2.IMREAD_GRAYSCALE)
    img_b = cv2.imread(str(img_path_b), cv2.IMREAD_GRAYSCALE)

    if img_a is None or img_b is None:
        return None

    pts_a = cv2.goodFeaturesToTrack(img_a, maxCorners=max_corners, qualityLevel=0.01, minDistance=20)
    if pts_a is None:
        return None

    pts_b, status, err = cv2.calcOpticalFlowPyrLK(img_a, img_b, pts_a, None)
    good_a = pts_a[status.flatten() == 1]
    good_b = pts_b[status.flatten() == 1]

    if len(good_a) < 20:
        return None

    # 2D Affine Matrix Estimation Using RANSAC (Includes Scaling, Rotation, and Translation)
    M, inlier_mask = cv2.estimateAffinePartial2D(good_a, good_b, method=cv2.RANSAC, ransacReprojThreshold=3.0)

    if M is None:
        return None

    # If the inlier ratio is very low, the data is unreliable
    if inlier_mask.sum() / len(good_a) < 0.4:
        return None

    return M # 2x3 affine transformation matrix

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import cv2
from tqdm.auto import tqdm
import time

# GPU Hardware Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Active Hardware: {device}")

def pitch_at(frame, knot_frames, knot_pitches):
    return np.interp(frame, knot_frames, knot_pitches)

# Settings
OPTICAL_FLOW_STEP = 5
stabilized_positions = []
image_cache = {}

def get_gray_tensor(frame_no):
    """Read the image and send it to the GPU's VRAM"""
    if frame_no in image_cache:
        return image_cache[frame_no]

    file_path = Path(FRAMES_DIR) / f"frame_{str(int(frame_no)).zfill(5)}.jpg"
    img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)

    # Clearing the cache to prevent memory bloat
    if len(image_cache) > 15:
        image_cache.clear()

    if img is not None:
        # GPU offloading process
        img_tensor = torch.from_numpy(img).to(device).float()
        image_cache[frame_no] = (img, img_tensor)
        return img, img_tensor
    return None, None

# Calculate the total workload
total_tracks = df_geo["track_id"].nunique()
start_time = time.time()

# Loop with a Live Progress Bar
with tqdm(total=total_tracks, desc="Chicken Trajectories Are Being Stabilized") as pbar:
    for tid, grp in df_geo.groupby("track_id"):
        grp = grp.sort_values("frame").reset_index(drop=True)
        if len(grp) < 2:
            pbar.update(1)
            continue

        # Assign the first frame to the world coordinate system (origin)
        first_row = grp.iloc[0]
        lat_world, lng_world = pixel_to_latlng(first_row["base_px"], first_row["base_py"], first_row["frame"])
        east_world, north_world = latlng_to_local_m(lat_world, lng_world, REF_LAT, REF_LNG)

        stabilized_positions.append({
            "track_id": tid, "frame": first_row["frame"], "time_s": first_row["time_s"],
            "lat": lat_world, "lon": lng_world, "east": east_world, "north": north_world
        })

        curr_east, curr_north = east_world, north_world
        i = 0

        while i < len(grp) - 1:
            step = min(OPTICAL_FLOW_STEP, len(grp) - 1 - i)
            row_a = grp.iloc[i]
            row_b = grp.iloc[i + step]

            img_a_cpu, img_a_gpu = get_gray_tensor(row_a['frame'])
            img_b_cpu, img_b_gpu = get_gray_tensor(row_b['frame'])

            M = None
            if img_a_cpu is not None and img_b_cpu is not None:
                pts_a = cv2.goodFeaturesToTrack(img_a_cpu, maxCorners=150, qualityLevel=0.02, minDistance=20)
                if pts_a is not None:
                    pts_b, status, _ = cv2.calcOpticalFlowPyrLK(img_a_cpu, img_b_cpu, pts_a, None)
                    if pts_b is not None:
                        good_a = pts_a[status.flatten() == 1].squeeze()
                        good_b = pts_b[status.flatten() == 1].squeeze()

                        if len(good_a) >= 10 and good_a.ndim == 2:
                            M, _ = cv2.estimateAffinePartial2D(good_a, good_b, method=cv2.RANSAC, ransacReprojThreshold=3.0)

            # Performing Geometric Projection on GPU Cores
            if M is not None:
                # Loading the transformation matrix onto the GPU as a tensor
                M_tensor = torch.from_numpy(M).to(device).float()
                pt_a = torch.tensor([row_a["base_px"], row_a["base_py"], 1.0], device=device).float()

                # Matrix multiplication on a GPU
                pred_pt_b = torch.matmul(M_tensor, pt_a)

                # With .item(), return the result to the CPU based on the coordinates
                chicken_pixel_dx = row_b["base_px"] - pred_pt_b[0].item()
                chicken_pixel_dy = row_b["base_py"] - pred_pt_b[1].item()
            else:
                chicken_pixel_dx = row_b["base_px"] - row_a["base_px"]
                chicken_pixel_dy = row_b["base_py"] - row_a["base_py"]

            # Drone Telemetry Scaling
            _, _, h_b, _ = get_drone_state(row_b["frame"])
            pitch_b = pitch_at(row_b["frame"], KNOT_FRAMES, KNOT_PITCHES)
            distance_meters_per_pixel = (h_b / np.sin(np.radians(pitch_b))) / FOCAL_PX

            total_delta_east = chicken_pixel_dx * distance_meters_per_pixel
            total_delta_north = -chicken_pixel_dy * distance_meters_per_pixel

            # Interpolate the sub-steps and add them to the list
            for sub_step in range(1, step + 1):
                sub_row = grp.iloc[i + sub_step]
                weight = sub_step / step

                frame_east = curr_east + (total_delta_east * weight)
                frame_north = curr_north + (total_delta_north * weight)

                lat_new, lng_new = local_m_to_latlng(frame_east, frame_north, REF_LAT, REF_LNG)

                stabilized_positions.append({
                    "track_id": tid, "frame": sub_row["frame"], "time_s": sub_row["time_s"],
                    "lat": lat_new, "lon": lng_new, "east": frame_east, "north": frame_north
                })

            curr_east += total_delta_east
            curr_north += total_delta_north
            i += step

        # Update the bar
        pbar.update(1)

df_stabilized = pd.DataFrame(stabilized_positions)
df_stabilized.to_csv("/kaggle/working/chicken_positions_stabilized.csv", index=False)
print(f"\n The GPU-accelerated process is complete! Total time: {time.time() - start_time:.2f} seconds.")

In [ ]:
# Upload Optically Stabilized CSV Data
STABILIZED_CSV_PATH = "/kaggle/working/chicken_positions_stabilized.csv"

if os.path.exists(STABILIZED_CSV_PATH):
    df_stabilized = pd.read_csv(STABILIZED_CSV_PATH)

    # Using telemetry data, extract the drone's flight path and add it to the map as a base layer
    east_t, north_t = latlng_to_local_m(telemetry["lat"].values, telemetry["lng"].values, REF_LAT, REF_LNG)

    fig, ax = plt.subplots(figsize=(9, 9))
    # Trace the drone's path
    ax.plot(east_t, north_t, "-", color="lightgray", linewidth=1.5, label="dron rotası")

    # Have the stabilized chicken rotations plotted one by one
    for tid, grp in df_stabilized.groupby("track_id"):
        e = grp["east"].values
        n = grp["north"].values
        ax.plot(e, n, "o-", markersize=2.5, linewidth=1, alpha=0.8)

    ax.set_title("Chicken Trajectories After Optical Stabilization (Actual Ground Motion)")
    ax.set_xlabel("East, m")
    ax.set_ylabel("North, m")
    ax.set_aspect("equal")
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("/kaggle/working/validation_stabilized.png", dpi=120)
    plt.show()
else:
    print(f"No stabilized data found: {STABILIZED_CSV_PATH}")

# CALCULATION OF SPEED STATISTICS BASED ON STABILIZED DATA
def smooth_track_local(df_track, window=15):
    """Applies a rolling median filter to local meter coordinates."""
    df_track = df_track.sort_values("frame").reset_index(drop=True)
    df_track["east_smooth"] = df_track["east"].rolling(window, center=True, min_periods=1).median()
    df_track["north_smooth"] = df_track["north"].rolling(window, center=True, min_periods=1).median()
    return df_track

speeds_stabilized = []

for tid, grp in df_stabilized.sort_values("time_s").groupby("track_id"):
    if len(grp) < 2:
        continue
    # Median softening is being performed along the route
    grp = smooth_track_local(grp, window=15)

    e = grp["east_smooth"].values
    n = grp["north_smooth"].values
    dt = np.diff(grp["time_s"].values)

    dist = np.hypot(np.diff(e), np.diff(n))
    valid = dt > 0
    speeds_stabilized.extend((dist[valid] / dt[valid]).tolist())

print(f"\nSPEED STATISTICS:")
print(f"Median Speed = {np.median(speeds_stabilized):.2f} m/s")
print(f"95% Speed    = {np.percentile(speeds_stabilized, 95):.2f} m/s")

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import cv2
from tqdm.auto import tqdm
import time

# GPU Hardware Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Settings
OPTICAL_FLOW_STEP = 5
image_cache = {}
stabilized_positions = []

def get_gray_tensor(frame_no):
    """Read the image and send it to the GPU's VRAM"""
    if frame_no in image_cache:
        return image_cache[frame_no]

    file_path = Path(FRAMES_DIR) / f"frame_{str(int(frame_no)).zfill(5)}.jpg"
    img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)

    if len(image_cache) > 15:
        image_cache.clear()

    if img is not None:
        img_tensor = torch.from_numpy(img).to(device).float()
        image_cache[frame_no] = (img, img_tensor)
        return img, img_tensor
    return None, None

# Calculate the total workload
total_tracks = df_geo["track_id"].nunique()
start_time = time.time()

# Drone Ray Tracing Model + Optical Flow Integration
with tqdm(total=total_tracks, desc="Telemetry + Gimbal Optical Flow Filter") as pbar:
    for tid, grp in df_geo.groupby("track_id"):
        grp = grp.sort_values("frame").reset_index(drop=True)

        i = 0
        while i < len(grp):
            row_curr = grp.iloc[i]
            curr_frame = row_curr["frame"]

            # REMOVING GIMBAL JITTER USING OPTICAL FLOW
            clean_px = row_curr["base_px"]
            clean_py = row_curr["base_py"]

            if i > 0:
                step_back = min(OPTICAL_FLOW_STEP, i)
                row_prev = grp.iloc[i - step_back]

                img_prev_cpu, _ = get_gray_tensor(row_prev['frame'])
                img_curr_cpu, _ = get_gray_tensor(curr_frame)

                if img_prev_cpu is not None and img_curr_cpu is not None:
                    # Capture fixed ground points
                    pts_prev = cv2.goodFeaturesToTrack(img_prev_cpu, maxCorners=150, qualityLevel=0.02, minDistance=20)
                    if pts_prev is not None:
                        pts_curr, status, _ = cv2.calcOpticalFlowPyrLK(img_prev_cpu, img_curr_cpu, pts_prev, None)
                        if pts_curr is not None:
                            good_prev = pts_prev[status.flatten() == 1].squeeze()
                            good_curr = pts_curr[status.flatten() == 1].squeeze()

                            if len(good_prev) >= 10 and good_prev.ndim == 2:
                                # The camera motion matrix (M) between two frames
                                M, _ = cv2.estimateAffinePartial2D(good_prev, good_curr, method=cv2.RANSAC, ransacReprojThreshold=3.0)

                                if M is not None:
                                    M_tensor = torch.from_numpy(M).to(device).float()
                                    pt_prev_tensor = torch.tensor([row_prev["base_px"], row_prev["base_py"], 1.0], device=device).float()

                                    # Where gimbal shake should be included based on optical flow
                                    pred_pt_curr = torch.matmul(M_tensor, pt_prev_tensor)
                                    expected_px = pred_pt_curr[0].item()
                                    expected_py = pred_pt_curr[1].item()

                                    # Pure (gimbal-free) pixel displacement of the chicken
                                    pixel_dx_pure_chicken = row_curr["base_px"] - expected_px
                                    pixel_dy_pure_chicken = row_curr["base_py"] - expected_py

                                    # Add only the chicken's own movement to the previously cleaned pixel
                                    clean_px = stabilized_positions[-1]["clean_px"] + pixel_dx_pure_chicken
                                    clean_py = stabilized_positions[-1]["clean_py"] + pixel_dy_pure_chicken

            # PROJECTION OF GEODETIC MODEL onto the Earth's Surface
            # Interpolating the drone's real-time telemetry status at that frame
            lat_d, lng_d, alt_d, yaw_d = get_drone_state(curr_frame)
            drone_east, drone_north = latlng_to_local_m(lat_d, lng_d, REF_LAT, REF_LNG)

            # Remove the pitch value that has been set as a constant from the function
            pitch_d = pitch_at(curr_frame, KNOT_FRAMES, KNOT_PITCHES)

            # The gimbal-stabilized pixels are fed into "project_ray_to_height" model
            coords = project_ray_to_height(
                drone_east=drone_east,
                drone_north=drone_north,
                height=alt_d,
                yaw_deg=yaw_d,
                pitch_down_deg=pitch_d,
                focal_px=FOCAL_PX,
                px=clean_px,
                py=clean_py,
                target_height=0.0
            )

            if coords is not None:
                east_world, north_world = coords[0], coords[1]
                lat_world, lng_world = local_m_to_latlng(east_world, north_world, REF_LAT, REF_LNG)
            else:
                # If the ray falls above the horizon (as a margin of error), keep the old coordinate
                east_world, north_world = drone_east, drone_north
                lat_world, lng_world = lat_d, lng_d

            # Save the results to the list
            stabilized_positions.append({
                "track_id": tid,
                "frame": curr_frame,
                "time_s": row_curr["time_s"],
                "clean_px": clean_px,
                "clean_py": clean_py,
                "lat": lat_world,
                "lon": lng_world,
                "east": east_world,
                "north": north_world
            })

            i += 1

        pbar.update(1)

df_stabilized = pd.DataFrame(stabilized_positions)
df_stabilized.drop(columns=["clean_px", "clean_py"], errors="ignore").to_csv("/kaggle/working/chicken_positions_stabilized.csv", index=False)
print(f"\n The beam model and optical gimbal stabilization are finished. Total time: {time.time() - start_time:.2f} seconds.")